# 02 — Dataset Understanding

**AttriSense · Employee Attrition Prediction & Analytics System**

Before cleaning or modeling, we need to know what the data contains: row count, column types, missing values, class balance, and columns that carry no information.

This notebook is read-only exploration. Transformations happen in `03_data_cleaning.ipynb`.

In [ ]:
# Standard imports for exploration.
# The attrisense package handles path resolution so notebooks stay portable.

import pandas as pd
import matplotlib.pyplot as plt

from attrisense.config import load_config
from attrisense.data import load_raw_data, dataset_summary

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

config = load_config()
RANDOM_STATE = config.random_state

## 1. Load Raw Data

The IBM HR Analytics dataset ships as a single CSV. We load it through the shared `load_raw_data()` function so every notebook reads from the same path defined in `configs/config.yaml`.

In [ ]:
df = load_raw_data()

summary = dataset_summary(df)
print(f"Rows    : {summary['rows']}")
print(f"Columns : {summary['columns']}")
print(f"Memory  : {summary['memory_mb']} MB")
print(f"Duplicate rows: {summary['duplicate_rows']}")

df.head()

## 2. Schema and Data Types

Understanding which columns are numeric vs categorical drives every downstream decision — encoding strategy, scaling, and which plots to use.

In [ ]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "unique": df.nunique(),
})

schema

### Column Groups

Grouping columns by domain makes the feature space easier to reason about:

| Group | Columns |
|-------|---------|
| **Target** | `Attrition` |
| **Identifier** | `EmployeeNumber` |
| **Demographics** | `Age`, `Gender`, `MaritalStatus`, `DistanceFromHome`, `Education`, `EducationField` |
| **Job / Role** | `Department`, `JobRole`, `JobLevel`, `JobInvolvement`, `BusinessTravel`, `OverTime` |
| **Compensation** | `MonthlyIncome`, `DailyRate`, `HourlyRate`, `MonthlyRate`, `PercentSalaryHike`, `StockOptionLevel` |
| **Tenure** | `TotalWorkingYears`, `YearsAtCompany`, `YearsInCurrentRole`, `YearsSinceLastPromotion`, `YearsWithCurrManager`, `NumCompaniesWorked` |
| **Satisfaction / Rating** | `EnvironmentSatisfaction`, `JobSatisfaction`, `RelationshipSatisfaction`, `WorkLifeBalance`, `PerformanceRating` |
| **Training** | `TrainingTimesLastYear` |
| **Constants (no variance)** | `EmployeeCount`, `Over18`, `StandardHours` |

## 3. Missing Values

Missing data would require imputation strategy. We check early because it affects the cleaning notebook.

In [ ]:
missing = df.isna().sum()
missing[missing > 0]

The dataset has **zero missing values** across all 35 columns. That simplifies cleaning — we do not need imputation, but we still need to drop constant columns and validate target labels.

## 4. Target Variable — Class Balance

Attrition datasets are typically imbalanced. The ratio between stayed and left employees determines which evaluation metrics matter and whether class weighting or resampling is needed during modeling.

In [ ]:
target_counts = df[config.target_column].value_counts()
target_pct = df[config.target_column].value_counts(normalize=True) * 100

print("Counts:\n", target_counts)
print("\nPercent:\n", target_pct.round(2))

fig, ax = plt.subplots(figsize=(5, 3.5))
target_counts.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"])
ax.set_title("Attrition Class Distribution")
ax.set_xlabel("Attrition")
ax.set_ylabel("Count")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

Roughly **16% attrition / 84% retention**. This imbalance confirms the metric choices from the problem understanding notebook — accuracy is not sufficient on its own.

## 5. Constant Columns

Columns with a single unique value cannot help a model discriminate between classes. We flag them here; the cleaning notebook removes them via `configs/config.yaml`.

In [ ]:
constant_cols = [col for col in df.columns if df[col].nunique() == 1]

pd.DataFrame({
    "column": constant_cols,
    "value": [df[col].iloc[0] for col in constant_cols],
})

## 6. Numeric Feature Ranges

Quick look at descriptive statistics for numeric columns. Outliers and skew will be explored further in the EDA notebook.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
df[numeric_cols].describe().T

## 7. Categorical Cardinality

Low-cardinality categoricals (Department, OverTime) are straightforward to encode. JobRole and EducationField have more levels — we note the counts here for the feature engineering phase.

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()

cardinality = (
    pd.Series({col: df[col].nunique() for col in categorical_cols}, name="unique_values")
    .sort_values(ascending=False)
)
cardinality

## 8. Key Observations (Summary)

Findings that feed into the next notebooks:

1. **1,470 rows, 35 columns** — modest size; overfitting risk is manageable with proper cross-validation.
2. **No missing values** — no imputation required.
3. **Class imbalance (~16% Yes)** — use recall, precision, F1, and ROC-AUC; consider `class_weight` in sklearn models.
4. **Three constant columns** — `EmployeeCount`, `Over18`, `StandardHours` will be dropped during cleaning.
5. **`EmployeeNumber` is a unique identifier** — excluded from features, kept for traceability.
6. **Mix of ordinal (1–4 satisfaction scales) and nominal categoricals** — encoding strategy matters; handled in feature engineering.

**Next step:** `03_data_cleaning.ipynb`